# Stream B — 03 Stage 2 OOD Classifier

This notebook trains/evaluates a Stage 2 classifier using:
- CLIP-derived features
- Stage 1 score
- local-image indicators

It evaluates on:
- NewsCLIPpings test
- MMFakeBench test
- VERITE local-image rows


In [1]:
from pathlib import Path
import sys, os, json, pickle, time, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root. Run this notebook inside eve-rag.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    MANIFESTS, PROCESSED_IMAGES, STREAM_B_CACHE,
    TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST,
    VERITE_TEST_MANIFEST, MMFAKEBENCH_VAL_MANIFEST, MMFAKEBENCH_TEST_MANIFEST,
    ALL_MANIFEST, FIGURES, TABLES
)

STREAM_B_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("STREAM_B_CACHE:", STREAM_B_CACHE)


ROOT: E:\eve-rag
STREAM_B_CACHE: E:\eve-rag\cache\stream_b


In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

# Tuned for 8 GB RTX 4060 Ti. Reduce if OOM.
IMAGE_BATCH_SIZE = 48
TEXT_BATCH_SIZE = 256
TRAIN_BATCH_SIZE = 512
NUM_WORKERS = 0  # Windows-friendly. Try 2 later if stable.
print("IMAGE_BATCH_SIZE:", IMAGE_BATCH_SIZE)
print("TEXT_BATCH_SIZE:", TEXT_BATCH_SIZE)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0
IMAGE_BATCH_SIZE: 48
TEXT_BATCH_SIZE: 256
TRAIN_BATCH_SIZE: 512


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

FEATURE_DIR = STREAM_B_CACHE / "features"
STAGE1_DIR = STREAM_B_CACHE / "lora_stage1"
STAGE2_DIR = STREAM_B_CACHE / "lora_stage2"
FINAL_DIR = STREAM_B_CACHE / "final"
for p in [STAGE2_DIR, FINAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

features = pd.read_csv(FEATURE_DIR / "clip_features_all.csv", low_memory=False)
stage1 = pd.read_csv(STAGE1_DIR / "stage1_scores_all.csv", low_memory=False)

df = features.merge(stage1[["pair_id", "stage1_score"]], on="pair_id", how="left")
df["stage1_score"] = df["stage1_score"].fillna(0.5)

print(df.shape)
df.groupby(["dataset", "split"]).size()


(97361, 14)


dataset        split
mmfakebench    test     10000
               val       1000
newsclippings  test      7264
               train    71072
               val       7024
verite         test      1001
dtype: int64

In [4]:
FEATURE_COLS = [
    "clip_cosine", "l2_dist", "absdiff_mean", "absdiff_max",
    "prod_mean", "text_norm", "image_norm", "has_real_image_emb", "stage1_score"
]

train_df = df[(df["dataset"] == "newsclippings") & (df["split"] == "train")].copy()
val_df = df[(df["dataset"] == "newsclippings") & (df["split"] == "val")].copy()
test_nc = df[(df["dataset"] == "newsclippings") & (df["split"] == "test")].copy()
test_mmfake = df[(df["dataset"] == "mmfakebench") & (df["split"] == "test")].copy()
test_verite = df[(df["dataset"] == "verite") & (df["has_image"] == 1)].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test_nc:", test_nc.shape)
print("test_mmfake:", test_mmfake.shape)
print("test_verite:", test_verite.shape)


train: (71072, 14)
val: (7024, 14)
test_nc: (7264, 14)
test_mmfake: (10000, 14)
test_verite: (621, 14)


In [5]:
X_train = train_df[FEATURE_COLS].fillna(0)
y_train = train_df["label"].astype(int)
X_val = val_df[FEATURE_COLS].fillna(0)
y_val = val_df["label"].astype(int)

# CPU model but fast. For these low-dimensional features, GPU would not help.
clf = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.01,
    random_state=42,
)
clf.fit(X_train, y_train)

pickle.dump({"model": clf, "feature_cols": FEATURE_COLS}, open(STAGE2_DIR / "stage2_model.pkl", "wb"), protocol=4)
print("Stage 2 model trained.")


Stage 2 model trained.


In [6]:
def get_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def metrics(y, score):
    pred = (score >= 0.5).astype(int)
    out = {
        "acc": accuracy_score(y, pred) * 100,
        "precision": precision_score(y, pred, zero_division=0) * 100,
        "recall": recall_score(y, pred, zero_division=0) * 100,
        "f1": f1_score(y, pred, zero_division=0) * 100,
        "auroc": roc_auc_score(y, score) * 100 if len(np.unique(y)) > 1 else np.nan,
    }
    return {k: round(v, 2) if not pd.isna(v) else v for k, v in out.items()}

rows = []
preds = []

for name, sub in [
    ("newsclippings_val", val_df),
    ("newsclippings_test", test_nc),
    ("mmfakebench_test", test_mmfake),
    ("verite_test_local", test_verite),
]:
    X = sub[FEATURE_COLS].fillna(0)
    y = sub["label"].astype(int).values
    score = get_score(clf, X)
    pred = (score >= 0.5).astype(int)
    rows.append({"dataset": name, **metrics(y, score)})
    tmp = sub[["pair_id", "dataset", "split", "label"]].copy()
    tmp["stream_b_score"] = score
    tmp["stream_b_pred"] = pred
    tmp["eval_dataset"] = name
    preds.append(tmp)

results = pd.DataFrame(rows)
pred_df = pd.concat(preds, ignore_index=True)

results.to_csv(TABLES / "stream_b_results.csv", index=False)
pred_df.to_csv(FINAL_DIR / "stream_b_predictions.csv", index=False)

results


,dataset,acc,precision,recall,f1,auroc
0,newsclippings_val,66.22,67.97,61.33,64.48,73.42
1,newsclippings_test,65.87,67.60,60.96,64.11,73.91
2,mmfakebench_test,46.95,82.38,30.80,44.84,63.35
3,verite_test_local,63.77,66.32,92.49,77.25,49.12


In [7]:
# Also save a compact final feature file for fusion.
fusion_b = df[["pair_id", "dataset", "split", "label"] + FEATURE_COLS].merge(
    pred_df[["pair_id", "stream_b_score", "stream_b_pred"]],
    on="pair_id",
    how="left"
)
fusion_b.to_csv(FINAL_DIR / "stream_b_final_features.csv", index=False)
fusion_b.head()


,pair_id,dataset,split,label,clip_cosine,l2_dist,absdiff_mean,absdiff_max,prod_mean,text_norm,image_norm,has_real_image_emb,stage1_score,stream_b_score,stream_b_pred
0,nc_620701_0,newsclippings,train,0,0.323501,1.163184,0.036120,0.479908,0.000632,1.0,1.0,1,0.3328,NaN,NaN
1,nc_620701_1,newsclippings,train,1,0.239942,1.232930,0.038847,0.501216,0.000469,1.0,1.0,1,0.8140,NaN,NaN
2,nc_1569820_0,newsclippings,train,0,0.240531,1.232452,0.034865,0.441948,0.000470,1.0,1.0,1,0.6416,NaN,NaN
3,nc_1569820_1,newsclippings,train,1,0.251289,1.223692,0.033795,0.444388,0.000491,1.0,1.0,1,0.5723,NaN,NaN
4,nc_1490080_0,newsclippings,train,0,0.302850,1.180805,0.035925,0.440455,0.000592,1.0,1.0,1,0.4860,NaN,NaN


In [8]:
print("=" * 60)
print("Stage 2 complete.")
print("Next: stream_b_results_fixed_updated.ipynb")
print("=" * 60)


Stage 2 complete.
Next: stream_b_results_fixed_updated.ipynb
